# Gig Worker Investment Advisor
**Upload the investments CSV when prompted, answer 5 quick questions, and receive your top 3 personalised investment recommendations with specific rupee projections and step-by-step guidance.**

In [3]:
# -------------------------------------------------------------------------
# CELL 1 - DATA LOADING
# -------------------------------------------------------------------------
# Loads the investments dataset (CSV) containing 36 investment options
# available in India.  Each row describes one option with the following
# information: name, type, expected return, risk level, liquidity, minimum
# investment amount, lock-in period, suitability description, whether it
# works for high-income-volatility users, and three numeric eligibility
# thresholds (minimum earning, maximum debt, minimum savings).
#
# On Google Colab : a file-upload dialog appears automatically.
# Running locally : set csv_filename below to your file path.
# -------------------------------------------------------------------------

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import files
    print("Running on Google Colab.")
    print("Upload your investments CSV file when the dialog appears below.")
    uploaded = files.upload()
    csv_filename = list(uploaded.keys())[0]
except ImportError:
    csv_filename = "investments.csv"   # <-- change this if running locally
    print("Running locally. Reading from:", csv_filename)

df_raw = pd.read_csv(csv_filename, on_bad_lines='skip')

print(f"\nDataset loaded successfully.")
print(f"Total investment options available: {len(df_raw)}")
print("Columns:", df_raw.columns.tolist())


Running locally. Reading from: investments.csv


FileNotFoundError: [Errno 2] No such file or directory: 'investments.csv'

In [ ]:
# -------------------------------------------------------------------------
# CELL 2 - DATA PREPARATION
# -------------------------------------------------------------------------
# Cleans and standardises the raw CSV columns for use in the scoring logic.
#
# Transformations applied:
#   Risk_Level text  -> Risk_Score integer (Very Low=1, Low=2,
#                       Low-Medium=2, Medium=3, High=4)
#   Liquidity text   -> Liquidity_Score integer (Low=1, Medium=2, High=3).
#                       Any non-standard text (e.g. "100 SIP") -> 2.
#   Safe_For_High_Volatility -> boolean Volatile_Safe
#   Gig_Priority NaN -> filled with 3 (neutral)
#   Numeric filter columns coerced to clean integers
#   Expected_Return string -> Return_Pct float (annual %)
#     - Range strings like "9-11% p.a." are converted to their midpoint (10%)
#     - Non-numeric strings like "Gold price linked" -> None
# -------------------------------------------------------------------------

import re

df = df_raw.copy()

# Risk Score
risk_map = {'Very Low': 1, 'Low': 2, 'Low-Medium': 2, 'Medium': 3, 'High': 4}
df['Risk_Score'] = df['Risk_Level'].str.strip().map(risk_map).fillna(2).astype(int)

# Liquidity Score
def map_liquidity(val):
    v = str(val).strip().lower()
    if v == 'low':    return 1
    if v == 'medium': return 2
    if v == 'high':   return 3
    return 2

df['Liquidity_Score'] = df['Liquidity'].apply(map_liquidity)

# Volatility safety flag
df['Volatile_Safe'] = df['Safe_For_High_Volatility'].apply(
    lambda x: str(x).strip().lower() == 'yes'
)

# Gig priority
df['Gig_Priority'] = pd.to_numeric(df['Gig_Priority'], errors='coerce').fillna(3).astype(int)

# Numeric eligibility columns
for col in ['Min_Monthly_Earning_Rs', 'Max_Debt_Rs', 'Min_Savings_Rs']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Parse numeric annual return from text
def parse_return(s):
    s = str(s)
    m = re.search(r'([\d.]+)\s*[-to]+\s*([\d.]+)\s*%', s)
    if m:
        return round((float(m.group(1)) + float(m.group(2))) / 2, 2)
    m = re.search(r'([\d.]+)\s*%', s)
    return float(m.group(1)) if m else None

df['Return_Pct'] = df['Expected_Return'].apply(parse_return)

print("Data preparation complete.")
print(f"Options with a parseable return figure: {df['Return_Pct'].notna().sum()} / {len(df)}")
print("\nSample check (key scoring columns):")
print(df[['Investment_Name','Risk_Score','Liquidity_Score',
          'Volatile_Safe','Gig_Priority','Return_Pct']].to_string())


Data preparation complete.
Options with a parseable return figure: 34 / 36

Sample check (key scoring columns):
                                                  Investment_Name  Risk_Score  Liquidity_Score  Volatile_Safe  Gig_Priority  Return_Pct
0                                     Post Office Savings Account           1                3           True             5        4.00
1                                 Post Office 1-Year Time Deposit           2                1           True             5        6.90
2                                 Post Office 2-Year Time Deposit           2                1           True             5        7.00
3                                 Post Office 3-Year Time Deposit           2                1           True             5        7.10
4                                 Post Office 5-Year Time Deposit           2                1          False             4        7.50
5                          Post Office Recurring Deposit (5-Year)       

In [ ]:
# -------------------------------------------------------------------------
# CELL 3 - USER QUESTIONNAIRE
# -------------------------------------------------------------------------
# Collects the 5 inputs needed to personalise the recommendation.
# All questions are multiple-choice (numbered options) so no free typing
# is required, which reduces errors.
#
# Inputs collected:
#   Q1. Monthly earning range   (4 brackets -> mapped to midpoint Rs value)
#   Q2. Total outstanding debt  (4 brackets -> mapped to midpoint Rs value)
#   Q3. Available savings       (numeric rupee input)
#   Q4. Income steadiness       (4 options -> volatility score 1-4)
#   Q5. Risk comfort level      (4 options -> risk tolerance score 1-4)
#
# All inputs are validated in a loop. The notebook will keep asking until
# a valid value is entered.
# -------------------------------------------------------------------------

def ask(prompt, options):
    print("\n" + prompt)
    for i, opt in enumerate(options, 1):
        print(f"  {i}. {opt}")
    while True:
        try:
            c = int(input("Your choice (enter the number): ").strip())
            if 1 <= c <= len(options):
                return c
            print(f"Please enter a number from 1 to {len(options)}.")
        except ValueError:
            print("Please type the number only, e.g. 2")

def ask_rupees(prompt):
    while True:
        try:
            v = int(input("\n" + prompt + "\n  Amount in Rupees (numbers only, e.g. 15000): ").strip())
            if v >= 0:
                return v
            print("Amount cannot be negative.")
        except ValueError:
            print("Please type a whole number without commas or symbols. Example: 15000")

print("=" * 62)
print("       GIG WORKER INVESTMENT ADVISOR - 5 QUICK QUESTIONS")
print("=" * 62)
print()
print("Answer 5 short questions about your financial situation.")
print("Based on your answers we will show you the 3 best investment")
print("options with specific numbers, projections, and next steps.")

# Q1
eq = ask(
    "Q1. What is your average monthly earning from all your gig work?",
    [
        "Below Rs 10,000 per month",
        "Rs 10,000 to Rs 25,000 per month",
        "Rs 25,001 to Rs 50,000 per month",
        "Above Rs 50,000 per month"
    ]
)
earning_mid = {1: 7000, 2: 17500, 3: 37500, 4: 65000}
user_earning = earning_mid[eq]

# Q2
dq = ask(
    "Q2. What is the total outstanding debt you owe right now\n"
    "    (personal loans, credit cards, buy-now-pay-later, etc.)?",
    [
        "No debt at all",
        "Less than Rs 20,000",
        "Rs 20,000 to Rs 1,00,000",
        "More than Rs 1,00,000"
    ]
)
debt_mid = {1: 0, 2: 10000, 3: 60000, 4: 150000}
user_debt = debt_mid[dq]

# Q3
user_savings = ask_rupees(
    "Q3. How much money do you currently have saved that you are\n"
    "    willing to invest (in your bank account or at home)?"
)

# Q4
vq = ask(
    "Q4. How steady is your income from month to month?",
    [
        "Very steady  - almost the same amount every month",
        "Fairly steady - small ups and downs, nothing extreme",
        "Unsteady     - some months good, some months very bad",
        "Very unsteady - I cannot predict next month at all"
    ]
)
user_volatility = vq   # 1=stable, 4=very volatile

# Q5
rq = ask(
    "Q5. If you invest Rs 10,000 today and it drops to Rs 8,500\n"
    "    next month before recovering, how would you feel?",
    [
        "Cannot afford any loss at all - full safety is my priority",
        "A little uncomfortable, but I can wait it out",
        "Fine - I care more about long-term growth than short-term dips",
        "Completely fine - I fully accept market ups and downs"
    ]
)
user_risk_tol = rq   # 1=very low, 4=high

print()
print("=" * 62)
print("All 5 answers recorded. Running recommendation engine...")
print("=" * 62)


       GIG WORKER INVESTMENT ADVISOR - 5 QUICK QUESTIONS

Answer 5 short questions about your financial situation.
Based on your answers we will show you the 3 best investment
options with specific numbers, projections, and next steps.

Q1. What is your average monthly earning from all your gig work?
  1. Below Rs 10,000 per month
  2. Rs 10,000 to Rs 25,000 per month
  3. Rs 25,001 to Rs 50,000 per month
  4. Above Rs 50,000 per month
Your choice (enter the number): 2

Q2. What is the total outstanding debt you owe right now
    (personal loans, credit cards, buy-now-pay-later, etc.)?
  1. No debt at all
  2. Less than Rs 20,000
  3. Rs 20,000 to Rs 1,00,000
  4. More than Rs 1,00,000
Your choice (enter the number): 2

Q3. How much money do you currently have saved that you are
    willing to invest (in your bank account or at home)?
  Amount in Rupees (numbers only, e.g. 15000): 29000

Q4. How steady is your income from month to month?
  1. Very steady  - almost the same amount every

In [ ]:
# -------------------------------------------------------------------------
# CELL 4 - RECOMMENDATION LOGIC
# -------------------------------------------------------------------------
# Scores every investment option against the user's profile.
#
# STAGE 1 - Hard Eligibility Filters (failing options are removed):
#   Min_Monthly_Earning_Rs  : user's estimated earning must meet threshold
#   Min_Savings_Rs          : user must have enough to start the option
#   Max_Debt_Rs             : user's debt must be at or below the ceiling
#   Volatile_Safe filter    : when volatility score >= 3 (unsteady income),
#                             only Volatile_Safe=True options are kept.
#                             If fewer than 3 options survive, the filter
#                             is relaxed and the full eligible set is used.
#
# STAGE 2 - Weighted Scoring (0-100+ scale, higher = better):
#   a) Risk alignment  (40 pts max) : exact match = 40 pts; each step
#      away from the target risk level loses 15 pts; 3+ steps = 0 pts.
#   b) Liquidity       (25 pts volatile / 20 pts stable) : scored
#      proportionally to Liquidity_Score (1=Low, 2=Med, 3=High).
#      Weighted higher when income is unsteady to prioritise access to cash.
#   c) Gig Priority    (25 pts max) : dataset's 1-5 suitability score
#      for gig workers, scaled linearly to 0-25 pts.
#   d) Volatility safety (15 pts)  : bonus only when income is unsteady
#      and the option is marked Volatile_Safe=True.
#
# Tiebreak: when two options have the same total score, the one with the
# higher Return_Pct is ranked first (more money for the same risk profile).
#
# Top 3 by total score (then return) are stored in `top3`.
# -------------------------------------------------------------------------

income_volatile = user_volatility >= 3

# Map user's risk tolerance (1-4) to the target Risk_Score (1-4)
target_risk = {1: 1, 2: 2, 3: 3, 4: 4}[user_risk_tol]

# Stage 1: Hard filters
base_mask = (
    (df['Min_Monthly_Earning_Rs'] <= user_earning) &
    (df['Min_Savings_Rs']         <= user_savings) &
    (df['Max_Debt_Rs']            >= user_debt)
)
eligible = df[base_mask].copy()

if income_volatile:
    vol_eligible = eligible[eligible['Volatile_Safe'] == True].copy()
    eligible = vol_eligible if len(vol_eligible) >= 3 else eligible

# Stage 2: Scoring
liq_weight = 25 if income_volatile else 20

def score(row):
    diff = abs(row['Risk_Score'] - target_risk)
    s  = [40, 25, 10, 0][min(diff, 3)]           # risk alignment
    s += (row['Liquidity_Score'] / 3) * liq_weight # liquidity
    s += (row['Gig_Priority'] - 1) * (25 / 4)     # gig priority
    if income_volatile and row['Volatile_Safe']:
        s += 15                                    # volatility safety bonus
    return round(s, 2)

eligible['Match_Score'] = eligible.apply(score, axis=1)

# Sort: primary = Match_Score descending, tiebreak = Return_Pct descending
# Use a NaN-safe tiebreak: fill None returns with 0 for sorting only
eligible['_sort_ret'] = eligible['Return_Pct'].fillna(0)
top3 = (eligible
        .sort_values(['Match_Score', '_sort_ret'], ascending=[False, False])
        .head(3)
        .reset_index(drop=True))

print(f"Eligible options after filtering : {len(eligible)}")
print(f"Top 3 selected:")
for i, row in top3.iterrows():
    print(f"  {i+1}. {row['Investment_Name']}  "
          f"(Score: {row['Match_Score']}, Return: {row['Return_Pct']}%)")
print("\nProceed to Cell 5 for your full personalised report.")


Eligible options after filtering : 34
Top 3 selected:
  1. ICICI Pru Ultra Short Term Fund  (Score: 85.0, Return: 7.0%)
  2. Edelweiss CRISIL-IBX AAA Bond Index Fund (Jun 2027)  (Score: 85.0, Return: 6.9%)
  3. Aditya Birla SL CRISIL-IBX AAA NBFC-HFC Index Fund (Sep 2026)  (Score: 85.0, Return: 6.9%)

Proceed to Cell 5 for your full personalised report.


In [ ]:
# -------------------------------------------------------------------------
# CELL 5 - PERSONALISED RECOMMENDATIONS WITH FULL NLP-STYLE EXPLANATION
# -------------------------------------------------------------------------
# Prints the top 3 recommendations in plain, conversational language.
# Every section references the user's exact numbers so the output reads
# as a personalised report, not a generic template.
#
# Structure of each recommendation:
#   WHAT IS THIS?               - plain-English description of the option
#   KEY NUMBERS                 - return %, risk level, liquidity, minimum
#                                 investment, lock-in period
#   YOUR PROJECTED GROWTH       - rupee value of their investable amount
#                                 after 1 year and 3 years at the stated
#                                 return rate (compound interest)
#   WHY THIS IS RIGHT FOR YOU   - personalised sentences linking each
#                                 scoring dimension to the user's answers
#   IF YOU NEED YOUR MONEY BACK - honest liquidity reality check
#   HOW TO GET STARTED          - app names / branch guidance with
#                                 minimum first deposit amount
#
# A 50-30-20 split suggestion is printed at the end using the user's
# exact investable amount across all three recommended options.
# -------------------------------------------------------------------------

import math

# ---- Label maps ----
risk_words = {
    1: "Very Low  - government-guaranteed, no realistic chance of loss",
    2: "Low       - bank or government backed, losing money is extremely unlikely",
    3: "Medium    - market-linked; short-term dips possible but usually recovers",
    4: "High      - market-linked; can drop 10-20% in a bad year"
}
liq_words = {
    1: "Low    - money is locked in for the full term; early exit is costly or not allowed",
    2: "Medium - exit is possible with a small penalty or a few days waiting period",
    3: "High   - withdraw any business day; money arrives in your account within 1-2 days"
}
earning_str  = {7000:"below Rs 10,000", 17500:"Rs 10,000-25,000",
                37500:"Rs 25,000-50,000", 65000:"above Rs 50,000"}
vol_str      = {1:"very steady", 2:"fairly steady", 3:"unsteady", 4:"very unsteady"}
risk_tol_str = {
    1: "want full capital safety with zero chance of loss",
    2: "can accept very small short-term dips as long as the principal is safe",
    3: "are comfortable with moderate market risk for better long-term returns",
    4: "fully accept higher market risk in exchange for maximum long-term growth"
}

# ---- Investable amount (savings minus a 2-month emergency buffer) ----
emergency_buffer = user_earning * 2
surplus = user_savings - emergency_buffer
if user_debt > user_savings * 0.5:
    surplus = surplus * 0.5          # if debt is heavy, invest only half the surplus
invest_amt = max(int(surplus), 0)

# ---- Compound growth helper ----
def project(amount, rate_pct, years):
    if amount <= 0 or rate_pct is None:
        return None
    return round(amount * ((1 + rate_pct / 100) ** years))

# ================================================================
# HEADER
# ================================================================
print("=" * 66)
print("         YOUR 3 PERSONALISED INVESTMENT RECOMMENDATIONS")
print("=" * 66)
print()
print("Here is a summary of your financial profile as we understood it:")
print()
print(f"  Monthly earning   :  {earning_str[user_earning]} per month")
print(f"  Current savings   :  Rs {user_savings:,}")
print(f"  Total debt owed   :  Rs {user_debt:,}")
print(f"  Income pattern    :  {vol_str[user_volatility]}")
print(f"  Risk comfort      :  You {risk_tol_str[user_risk_tol]}")
print()
print(f"  Emergency buffer  :  Rs {emergency_buffer:,}  (2 months income kept aside)")
print(f"  Amount to invest  :  Rs {invest_amt:,}  (savings after the buffer)")
print()

# ---- Debt warning ----
if user_debt > user_savings:
    print("-" * 66)
    print("IMPORTANT - DEBT NOTE")
    print(f"  Your outstanding debt of Rs {user_debt:,} is larger than your total")
    print(f"  savings of Rs {user_savings:,}.  High-interest debts (credit cards,")
    print(f"  personal loans above 12% per year) cost more than any investment")
    print(f"  below can earn.  Pay off the most expensive debts first, then")
    print(f"  invest the freed-up cash every month.  The 3 options below are")
    print(f"  still the right choices for whatever small surplus you can set aside.")
    print("-" * 66)
    print()

# ---- Zero savings note ----
if user_savings == 0:
    print("-" * 66)
    print("STARTING FROM ZERO")
    print("  You have no savings to invest right now. All 3 options below")
    print("  can be started with as little as Rs 1 to Rs 500.  Even setting")
    print("  aside Rs 100-200 per month builds the habit and compounds over time.")
    print("-" * 66)
    print()

print("The 3 options below were ranked from", len(df), "available options using")
print("your earning level, savings, debt, income pattern, and risk comfort.")

# ================================================================
# PER-RECOMMENDATION LOOP
# ================================================================
ordinal   = ["FIRST", "SECOND", "THIRD"]
rank_note = [
    "Best overall match for your profile.",
    "Strong second choice - ideal as a complement to Option 1.",
    "Third-best fit - useful for spreading across two or three options."
]

for i, row in top3.iterrows():
    r      = i + 1
    ret    = row['Return_Pct']
    rs     = row['Risk_Score']
    ls     = row['Liquidity_Score']
    lock   = str(row['Lock_in_Period']) if pd.notna(row['Lock_in_Period']) else "None"
    itype  = str(row['Type']).lower()
    iname  = row['Investment_Name']
    min_inv = row['Min_Investment_Rs']

    val1   = project(invest_amt, ret, 1)
    val3   = project(invest_amt, ret, 3)

    print()
    print("=" * 66)
    print(f"  OPTION {r}  ({ordinal[i]} RECOMMENDATION)")
    print(f"  {iname.upper()}")
    print(f"  {rank_note[i]}")
    print("=" * 66)

    # ---- WHAT IS THIS ----
    print()
    print("WHAT IS THIS?")
    print(f"  {row['Description']}")
    print(f"  Type: {row['Type']}")

    # ---- KEY NUMBERS ----
    print()
    print("KEY NUMBERS")
    print(f"  Expected annual return  :  {row['Expected_Return']}")
    if ret:
        print(f"  Return (numeric)        :  {ret}% per year")
    print(f"  Risk level              :  {risk_words[rs]}")
    print(f"  Liquidity               :  {liq_words[ls]}")
    print(f"  Minimum to start        :  Rs {min_inv}")
    print(f"  Lock-in period          :  {lock}")

    # ---- PROJECTED GROWTH ----
    print()
    print("YOUR PROJECTED GROWTH")
    if invest_amt > 0 and ret is not None:
        gain1 = val1 - invest_amt
        gain3 = val3 - invest_amt
        print(f"  If you invest Rs {invest_amt:,} at {ret}% per year:")
        print(f"  After 1 year  ->  Rs {val1:,}  (Rs {gain1:,} earned on top)")
        print(f"  After 3 years ->  Rs {val3:,}  (Rs {gain3:,} earned on top)")
        if rs >= 3:
            print(f"  Note: This is a market-linked option. In a bad year it could")
            print(f"  drop to roughly Rs {round(invest_amt * 0.85):,} before recovering.")
            print(f"  The figures above are long-run averages, not guarantees.")
        else:
            print(f"  This is a fixed-rate or government-backed option, so these")
            print(f"  figures are close to what you will actually receive.")
    elif invest_amt == 0 and ret is not None:
        eg = 5000
        eg1 = round(eg * (1 + ret/100))
        eg3 = round(eg * (1 + ret/100)**3)
        print(f"  You have no investable surplus right now, but once you save:")
        print(f"  Rs 5,000 invested at {ret}% per year -> Rs {eg1:,} after 1 year,")
        print(f"  Rs {eg3:,} after 3 years.")
    else:
        print(f"  This option is linked to the gold price, which changes daily.")
        print(f"  Gold has historically returned roughly 8-10% per year in India")
        print(f"  over long periods, but there are no guarantees.")

    # ---- WHY THIS IS RIGHT FOR YOU ----
    print()
    print("WHY THIS IS RIGHT FOR YOU")

    # Risk reasoning
    if rs == 1:
        print(f"  Your money is protected by the government - it literally cannot")
        print(f"  lose value.  Since you {risk_tol_str[user_risk_tol]}, this is the")
        print(f"  safest home for your money.  You will never open your account to")
        print(f"  find less money than you put in.")
    elif rs == 2:
        print(f"  This is a low-risk option backed by a bank or the government.")
        print(f"  Losing money here is extremely unlikely.  For someone earning")
        print(f"  {earning_str[user_earning]} per month, protecting existing savings")
        print(f"  is just as important as growing them.")
    elif rs == 3:
        print(f"  This option carries medium risk.  You said you {risk_tol_str[user_risk_tol]},")
        print(f"  which is exactly the right profile for this.  On any given day")
        print(f"  the value could be slightly up or down, but over 1-3 years it has")
        print(f"  consistently delivered positive returns above fixed deposits.")
    else:
        print(f"  This is a higher-risk investment.  You said you {risk_tol_str[user_risk_tol]},")
        print(f"  which makes you a good match.  In a bad year it can fall 10-20%,")
        print(f"  but historically over any 7+ year period it has returned 12-15%")
        print(f"  per year - the highest of any option in this list.")

    # Volatility reasoning
    if income_volatile and row['Volatile_Safe']:
        print(f"  Your income is {vol_str[user_volatility]}, so some months you may")
        print(f"  not be able to contribute at all.  This option was specifically chosen")
        print(f"  because it has no mandatory monthly commitment - you invest when you")
        print(f"  can and skip months when money is tight, without any penalty.")
    elif income_volatile and not row['Volatile_Safe']:
        print(f"  Your income is {vol_str[user_volatility]}.  Since this option has a")
        print(f"  lock-in period ({lock}), only put in money you are confident")
        print(f"  you will NOT need before the lock-in ends.")
    else:
        print(f"  Since your income is {vol_str[user_volatility]}, you can set up a fixed")
        print(f"  monthly auto-debit into this option and let it compound without")
        print(f"  thinking about it each month.")

    # Savings / debt context
    if user_savings > 0 and invest_amt > 0:
        leftover = user_savings - invest_amt
        print(f"  Investing Rs {invest_amt:,} here still leaves Rs {leftover:,} in your")
        print(f"  account as a liquid emergency fund.")
    elif user_savings > 0 and invest_amt == 0:
        print(f"  After keeping Rs {emergency_buffer:,} aside as an emergency buffer,")
        print(f"  you have no surplus to invest right now. Start with as little as")
        print(f"  Rs {min_inv} whenever you have extra money.  Every rupee counts.")

    # ---- LIQUIDITY REALITY CHECK ----
    print()
    print("IF YOU NEED YOUR MONEY BACK URGENTLY")
    if ls == 3:
        print(f"  You can withdraw on any business day.  The money arrives in your")
        print(f"  bank account within 1-2 working days.  No penalty, no waiting.")
        print(f"  This is the most flexible option for gig workers who may face")
        print(f"  sudden income drops.")
    elif ls == 2:
        print(f"  Lock-in: {lock}.  You can exit before maturity but you will")
        print(f"  receive a slightly lower return or pay a small exit charge.")
        print(f"  Do not put money here that you might urgently need within 1-2 months.")
    else:
        print(f"  Lock-in: {lock}.  Early withdrawal is not allowed or comes with a")
        print(f"  significant penalty.  Only invest money you are certain you will")
        print(f"  NOT need before this period ends.  Before investing here, make sure")
        print(f"  you have at least Rs {emergency_buffer:,} (2 months income) sitting")
        print(f"  in a liquid savings account outside this option.")

    # ---- HOW TO GET STARTED ----
    print()
    print("HOW TO GET STARTED")
    if 'government small savings' in itype or 'post office' in iname.lower():
        print(f"  Walk into any Post Office or a nationalised bank (SBI, Bank of Baroda,")
        print(f"  Canara Bank, PNB).  Ask for '{iname}' by that exact name.")
        print(f"  Bring: Aadhaar card, PAN card, one passport-size photo.")
        print(f"  Minimum first deposit: Rs {min_inv}.")
    elif 'mutual fund' in itype or 'etf' in itype:
        print(f"  Download Groww, Zerodha Coin, or Paytm Money on your phone.")
        print(f"  Complete a one-time KYC using Aadhaar + PAN (takes ~10 minutes).")
        print(f"  Search for '{iname}' and tap Invest.")
        print(f"  Minimum investment: Rs {min_inv}.  No branch visit needed.")
    elif 'nps' in itype or 'pension' in itype:
        print(f"  Open an NPS account at npscra.nsdl.co.in or visit any bank branch.")
        print(f"  Bring: Aadhaar, PAN, cancelled cheque.")
        print(f"  Minimum monthly contribution: Rs {min_inv}.")
    elif 'digital gold' in itype or ('commodity' in itype and 'digital' in iname.lower()):
        print(f"  Open Groww, PhonePe, or Paytm on your phone.")
        print(f"  Tap 'Gold' or search for 'Digital Gold'.  Start with as little as Rs 1.")
        print(f"  No physical storage or paperwork needed.")
    elif 'gold etf' in itype or ('etf' in itype and 'gold' in iname.lower()):
        print(f"  Open a Demat account on Zerodha, Groww, or Upstox (free to open).")
        print(f"  Search for '{iname}' and buy units at the current market price.")
        print(f"  Minimum: ~Rs {min_inv} (price of 1 unit, changes daily).")
    elif 'gold bond' in itype or 'sgb' in iname.lower() or 'sovereign gold' in iname.lower():
        print(f"  Buy during the RBI's open subscription window through your bank app")
        print(f"  or a broker like Zerodha or Groww.  Search 'Sovereign Gold Bond'.")
        print(f"  Minimum: 1 gram of gold (approximately Rs 5,000-6,500 depending on")
        print(f"  the gold price that week).  No physical gold is delivered.")
    elif 'bank fixed' in itype:
        print(f"  Log into your bank's mobile app (SBI YONO, HDFC MobileBanking, etc.)")
        print(f"  and select 'Fixed Deposit' from the menu.  Takes under 5 minutes.")
        print(f"  Or walk into any bank branch.  Minimum deposit: Rs {min_inv}.")
    else:
        print(f"  Visit a SEBI-registered financial advisor or your nearest bank branch.")
        print(f"  Mention '{iname}' by name and ask about all charges and the exact")
        print(f"  lock-in before signing anything.")

# ================================================================
# CLOSING: 50-30-20 SPLIT SUGGESTION
# ================================================================
print()
print("=" * 66)
print("  FINAL ADVICE - HOW TO SPLIT YOUR INVESTMENT")
print("=" * 66)
print()
print("You do not have to choose just one option. Splitting your money")
print("across two or three options balances safety, growth, and access")
print("to emergency cash at the same time.")
print()

n1 = top3['Investment_Name'].iloc[0]
n2 = top3['Investment_Name'].iloc[1]
n3 = top3['Investment_Name'].iloc[2]

if invest_amt > 0:
    s1 = round(invest_amt * 0.50)
    s2 = round(invest_amt * 0.30)
    s3 = invest_amt - s1 - s2
    print(f"  Suggested split of your Rs {invest_amt:,}:")
    print(f"  50%  =  Rs {s1:,}   ->  {n1}")
    print(f"  30%  =  Rs {s2:,}   ->  {n2}")
    print(f"  20%  =  Rs {s3:,}   ->  {n3}")
    print()
    # Project combined growth
    r1 = top3['Return_Pct'].iloc[0]
    r2 = top3['Return_Pct'].iloc[1]
    r3 = top3['Return_Pct'].iloc[2]
    if all(x is not None for x in [r1, r2, r3]):
        combined_1yr = (project(s1,r1,1) or 0) + (project(s2,r2,1) or 0) + (project(s3,r3,1) or 0)
        combined_3yr = (project(s1,r1,3) or 0) + (project(s2,r2,3) or 0) + (project(s3,r3,3) or 0)
        print(f"  If you split this way, your Rs {invest_amt:,} could grow to:")
        print(f"  Rs {combined_1yr:,} after 1 year  (Rs {combined_1yr - invest_amt:,} earned)")
        print(f"  Rs {combined_3yr:,} after 3 years (Rs {combined_3yr - invest_amt:,} earned)")
else:
    print(f"  Once you have surplus to invest, consider a 50-30-20 split:")
    print(f"  50%  ->  {n1}")
    print(f"  30%  ->  {n2}")
    print(f"  20%  ->  {n3}")

print()
print("One rule to always keep: maintain at least Rs", emergency_buffer,
      f"(2 months of")
print(f"your Rs {user_earning:,} income) in a regular savings account as an emergency")
print(f"fund at all times. Invest only money that is above this buffer.")
print()
print("Start small. Be consistent. Your savings will grow over time.")
print()


         YOUR 3 PERSONALISED INVESTMENT RECOMMENDATIONS

Here is a summary of your financial profile as we understood it:

  Monthly earning   :  Rs 10,000-25,000 per month
  Current savings   :  Rs 29,000
  Total debt owed   :  Rs 10,000
  Income pattern    :  fairly steady
  Risk comfort      :  You can accept very small short-term dips as long as the principal is safe

  Emergency buffer  :  Rs 35,000  (2 months income kept aside)
  Amount to invest  :  Rs 0  (savings after the buffer)

The 3 options below were ranked from 36 available options using
your earning level, savings, debt, income pattern, and risk comfort.

  OPTION 1  (FIRST RECOMMENDATION)
  ICICI PRU ULTRA SHORT TERM FUND
  Best overall match for your profile.

WHAT IS THIS?
  Ultra-short duration debt securities.
  Type: Mutual Fund - Ultra Short

KEY NUMBERS
  Expected annual return  :  ~7.0% p.a. (indicative)
  Return (numeric)        :  7.0% per year
  Risk level              :  Low       - bank or government backe

# Technical Documentation: Investment Suggestion Module

### 1. Module Overview
The Investment Suggestion Module is designed to provide personalized financial product recommendations for gig economy workers in the Indian market. It addresses the challenge of high income volatility and varying financial literacy by filtering a diverse dataset of 36 investment vehicles to identify the most suitable options based on an individual's unique cash flow and risk profile.

### 2. Input Parameters
*   **Financial Dataset (CSV):** Contains 36 investment options with attributes including Return Percentage, Risk Level (Categorical), Liquidity (Categorical), Minimum Investment (Integer), and Suitability Flags.
*   **User Earnings (Ordinal):** Average monthly income categorized into four brackets (Below ₹10k to Above ₹50k).
*   **Outstanding Debt (Ordinal):** Total liabilities categorized into four brackets.
*   **Available Savings (Integer):** Liquid capital available for investment.
*   **Income Stability (Ordinal):** A 1–4 scale measuring income predictability.
*   **Risk Tolerance (Ordinal):** A 1–4 scale measuring psychological comfort with capital drawdowns.

### 3. Data Processing & Logic
The module executes a two-stage pipeline:
1.  **Preprocessing & Standardization:**
    *   Categorical risk and liquidity strings are mapped to ordinal integer scales (1–4).
    *   Annualized return strings (e.g., "9-11%") are parsed using regex and converted to floating-point midpoints.
    *   Boolean flags are generated for 'Volatility Safety' based on the product's suitability for users with unpredictable income.
2.  **Filtering & Scoring:**
    *   **Hard Eligibility Filter:** Eliminates products where the user’s earnings or savings fall below the minimum threshold, or where debt exceeds the maximum allowable limit.
    *   **Weighted Scoring Engine:** Calculates a 'Match Score' based on:
        *   Risk Alignment (40%): Proximity to user's risk tolerance.
        *   Liquidity (20-25%): Prioritized higher for users with unsteady income.
        *   Gig Worker Priority (25%): Pre-defined suitability metrics.
        *   Volatility Bonus (15%): Awarded to stable products for unsteady earners.

### 4. Algorithm Used
The system utilizes a **Multi-Criteria Weighted Scoring Heuristic**. This rule-based approach was chosen over machine learning models to ensure complete transparency, explainability, and reliability in a high-stakes financial context where training data for this specific niche is limited.

### 5. Output
*   **Top 3 Recommendations:** A ranked list of investment products.
*   **Match Scores:** Normalized scores indicating the strength of the fit.
*   **Projected Growth:** 1-year and 3-year compound interest projections based on the user's investable surplus.
*   **Strategic Allocation:** A 50-30-20 portfolio split suggestion across the top three results.

### 6. Assumptions
*   **Inflation:** Projections assume constant returns and do not account for inflation or taxation.
*   **Emergency Fund:** The logic assumes a mandatory 2-month income buffer should be maintained before investing.
*   **Market Returns:** Historic or indicative returns are used as proxies for future performance.

### 7. Limitations
*   **Static Data:** The module relies on a static CSV and does not pull real-time market data or interest rate updates.
*   **Taxation:** Does not account for individual tax brackets or capital gains taxes (LTCG/STCG).
*   **Single-Country Focus:** The logic is strictly tuned to Indian financial instruments and regulations.

### 8. Future Improvements
*   **API Integration:** Integrating real-time NAV (Net Asset Value) and interest rate feeds.
*   **Expense Tracking:** Incorporating actual monthly expense data instead of relying on estimated income brackets.
*   **Tax Optimization:** Adding a logic layer to suggest tax-saving instruments (80C) based on the user's tax liability.